In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [65]:
df = pd.read_csv('imdb_top_1000.csv')

df = df[['Series_Title', 'Genre', 'Director', 'Star1', 'Star2', 'Star3', 'Star4']]
df['Series_Title'] = df['Series_Title'].str.lower()
df

,Series_Title,Genre,Director,Star1,Star2,Star3,Star4
0,the shawshank redemption,Drama,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler
1,the godfather,"Crime, Drama",Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton
2,the dark knight,"Action, Crime, Drama",Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine
3,the godfather: part ii,"Crime, Drama",Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton
4,12 angry men,"Crime, Drama",Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler
...,...,...,...,...,...,...,...
995,breakfast at tiffany's,"Comedy, Drama, Romance",Blake Edwards,Audrey Hepburn,George Peppard,Patricia Neal,Buddy Ebsen
996,giant,"Drama, Western",George Stevens,Elizabeth Taylor,Rock Hudson,James Dean,Carroll Baker
997,from here to eternity,"Drama, Romance, War",Fred Zinnemann,Burt Lancaster,Montgomery Clift,Deborah Kerr,Donna Reed
998,lifeboat,"Drama, War",Alfred Hitchcock,Tallulah Bankhead,John Hodiak,Walter Slezak,William Bendix


In [66]:
df['Combined_Features'] = df['Genre'] + ' ' + df['Director'] + ' ' + df['Star1'] + ' ' + df['Star2'] + ' ' + df['Star3'] + ' ' + df['Star4']

df

,Series_Title,Genre,Director,Star1,Star2,Star3,Star4,Combined_Features
0,the shawshank redemption,Drama,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,Drama Frank Darabont Tim Robbins Morgan Freeman Bob Gunton William Sadler
1,the godfather,"Crime, Drama",Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,"Crime, Drama Francis Ford Coppola Marlon Brando Al Pacino James Caan Diane Keaton"
2,the dark knight,"Action, Crime, Drama",Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,"Action, Crime, Drama Christopher Nolan Christian Bale Heath Ledger Aaron Eckhart Michael Caine"
3,the godfather: part ii,"Crime, Drama",Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,"Crime, Drama Francis Ford Coppola Al Pacino Robert De Niro Robert Duvall Diane Keaton"
4,12 angry men,"Crime, Drama",Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,"Crime, Drama Sidney Lumet Henry Fonda Lee J. Cobb Martin Balsam John Fiedler"
...,...,...,...,...,...,...,...,...
995,breakfast at tiffany's,"Comedy, Drama, Romance",Blake Edwards,Audrey Hepburn,George Peppard,Patricia Neal,Buddy Ebsen,"Comedy, Drama, Romance Blake Edwards Audrey Hepburn George Peppard Patricia Neal Buddy Ebsen"
996,giant,"Drama, Western",George Stevens,Elizabeth Taylor,Rock Hudson,James Dean,Carroll Baker,"Drama, Western George Stevens Elizabeth Taylor Rock Hudson James Dean Carroll Baker"
997,from here to eternity,"Drama, Romance, War",Fred Zinnemann,Burt Lancaster,Montgomery Clift,Deborah Kerr,Donna Reed,"Drama, Romance, War Fred Zinnemann Burt Lancaster Montgomery Clift Deborah Kerr Donna Reed"
998,lifeboat,"Drama, War",Alfred Hitchcock,Tallulah Bankhead,John Hodiak,Walter Slezak,William Bendix,"Drama, War Alfred Hitchcock Tallulah Bankhead John Hodiak Walter Slezak William Bendix"


In [67]:
vectorizer = CountVectorizer(stop_words='english')

features_vector = vectorizer.fit_transform(df['Combined_Features'])

features_vector

<1000x4318 sparse matrix of type '<class 'numpy.int64'>'
	with 12705 stored elements in Compressed Sparse Row format>

In [68]:
similarities = cosine_similarity(features_vector)

similarities

array([[1.        , 0.0836242 , 0.0836242 , ..., 0.0836242 , 0.17407766,
        0.        ],
       [0.0836242 , 1.        , 0.15384615, ..., 0.07692308, 0.08006408,
        0.07692308],
       [0.0836242 , 0.15384615, 1.        , ..., 0.07692308, 0.08006408,
        0.07692308],
       ...,
       [0.0836242 , 0.07692308, 0.07692308, ..., 1.        , 0.16012815,
        0.        ],
       [0.17407766, 0.08006408, 0.08006408, ..., 0.16012815, 1.        ,
        0.16012815],
       [0.        , 0.07692308, 0.07692308, ..., 0.        , 0.16012815,
        1.        ]])

In [70]:
def recommend_movie(movie_title, num_of_recommendations=3):
    movie_title = movie_title.lower()

    if len(df['Series_Title'][df['Series_Title'] == movie_title]) == 0:
        print("The Movie is not in dataset")

    idx = df[df['Series_Title'] == movie_title].index[0]

    similarity_score = list(enumerate(similarities[idx]))
    similarity_score = sorted(similarity_score, key=lambda x: x[1], reverse=True)

    recommended_indices = [i[0] for i in similarity_score[:num_of_recommendations+1]]
    recommended_score = [i[1]* 100 for i in similarity_score[:num_of_recommendations+1]]

    return np.c_[df['Series_Title'].iloc[recommended_indices[1:]].values, recommended_score[1:]]

In [71]:
for movie, similarity in recommend_movie('the godfather', 8):
    print(f"Movie : \n\t\t{movie}\nSimilarity score :  \n\t\t{similarity:0.2f}\n{'*'*50}")


Movie : 
		the godfather: part iii
Similarity score :  
		69.23
**************************************************
Movie : 
		the godfather: part ii
Similarity score :  
		64.45
**************************************************
Movie : 
		apocalypse now
Similarity score :  
		44.47
**************************************************
Movie : 
		glengarry glen ross
Similarity score :  
		38.46
**************************************************
Movie : 
		scent of a woman
Similarity score :  
		33.45
**************************************************
Movie : 
		scarface
Similarity score :  
		30.77
**************************************************
Movie : 
		heat
Similarity score :  
		30.77
**************************************************
Movie : 
		on the waterfront
Similarity score :  
		30.77
**************************************************
